# Bio-Inspired Edge Detection Models Comparison

This notebook aggregates results from all 12 traditional bio-inspired edge detection models (Table 1).

**Models**: Kim 2015 → Smith 2024 (12 studies)  
**Dataset**: HED_Small test set (20 images)  
**Metrics**: ODS, OIS, AP

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'seaborn'], check=False)
import json, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Model info
models_info = [
    ('Kim_2015', 'Kim et al. 2015', 'LGN+V1', 'DoG + 8-orient Gabor'),
    ('Patel_2018', 'Patel et al. 2018', 'LGN+V1', 'Multi-scale'),
    ('Li_2018', 'Li et al. 2018', 'LGN+V1', 'Adaptive DoG'),
    ('Lee_2019', 'Lee et al. 2019', 'LGN+V1+V2/V4+Multi', 'First full hierarchy'),
    ('Zhang_2019', 'Zhang et al. 2019', 'LGN+V1', '12 orientations'),
    ('Park_2020', 'Park et al. 2020', 'LGN+V1+Multi', 'Two-level fusion'),
    ('Chen_2020', 'Chen et al. 2020', 'LGN+V1+V2/V4+Multi', 'Complete hierarchy'),
    ('Nguyen_2020', 'Nguyen et al. 2020', 'LGN+V1+V2/V4+Multi', '16-orient feedback'),
    ('Wang_2020', 'Wang et al. 2020', 'LGN+V1+Multi', 'Hierarchical pooling'),
    ('Zhao_2022', 'Zhao et al. 2022', 'LGN+V1+V2/V4+Multi', 'Adaptive hierarchy'),
    ('Wu_2022', 'Wu et al. 2022', 'LGN+V1+Multi', 'Efficient'),
    ('Smith_2024', 'Smith et al. 2024', 'LGN+V1+V2/V4+Multi', 'SOTA 2024'),
]

# Aggregate results
results = []
for dirname, name, bio, desc in models_info:
    json_files = list((OUTPUT_DIR / dirname).glob('*.json')) if (OUTPUT_DIR / dirname).exists() else []
    if json_files:
        with open(json_files[0]) as f:
            data = json.load(f)
            m = data['metrics']
            results.append({
                'Model': name,
                'Bio Features': bio,
                'Description': desc,
                'ODS': m['ODS'],
                'OIS': m['OIS'],
                'AP': m['AP']
            })

df = pd.DataFrame(results)
print("\n📊 Bio-Inspired Models Results (Table 1)\n")
print(df.to_string(index=False))
print("\n" + "="*80)

In [ ]:
# Rankings
df_sorted = df.sort_values('ODS', ascending=False)
print("\n🏆 Rankings by ODS:\n")
for i, row in enumerate(df_sorted.itertuples(), 1):
    print(f"{i:2d}. {row.Model:25s} | ODS={row.ODS:.4f} | OIS={row.OIS:.4f} | AP={row.AP:.4f}")

# Statistics
print(f"\n📈 Statistics:")
print(f"   Mean ODS: {df['ODS'].mean():.4f} ± {df['ODS'].std():.4f}")
print(f"   Mean OIS: {df['OIS'].mean():.4f} ± {df['OIS'].std():.4f}")
print(f"   Mean AP:  {df['AP'].mean():.4f} ± {df['AP'].std():.4f}")
print(f"   Best:     {df_sorted.iloc[0]['Model']} (ODS={df_sorted.iloc[0]['ODS']:.4f})")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['ODS', 'OIS', 'AP']):
    df_plot = df.sort_values(metric, ascending=False)
    ax.barh(range(len(df_plot)), df_plot[metric], color='steelblue')
    ax.set_yticks(range(len(df_plot)))
    ax.set_yticklabels([m.split()[-1] for m in df_plot['Model']], fontsize=9)
    ax.set_xlabel(metric, fontsize=11, fontweight='bold')
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bio_comparison.png', dpi=150, bbox_inches='tight')
print(f"\n✅ Saved: {OUTPUT_DIR / 'bio_comparison.png'}")

# Save CSV
df.to_csv(OUTPUT_DIR / 'bio_results.csv', index=False)
print(f"✅ Saved: {OUTPUT_DIR / 'bio_results.csv'}\n")